# Mini Project 3 — Word Embeddings

This notebook implements a complete word embeddings pipeline for Khmer text:
- **Part I:** Skip-gram model with negative sampling
- **Part II:** PCA visualization of word embeddings
- **Part III:** Neural language model (next-word prediction)
- **Part IV:** Learning embeddings from scratch + comparison

## Setup & Imports

In [ ]:
import re
import json
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from khmernltk import word_tokenize

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from sklearn.decomposition import PCA

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Load Data

In [ ]:
# Load the text corpus
with open('temples.txt', 'r', encoding='utf-8') as f:
    raw_text = f.read()

print(f'Raw text length: {len(raw_text):,} characters')
print(f'First 200 chars: {raw_text[:200]}')

---
## Part I: Skip-gram Model with Negative Sampling

**Parameters:**
- Embedding dimension: 50
- Context window: ±4
- Negative sampling: 2
- Frequency threshold: ≥ 10

In [ ]:
# ---- 1. Tokenization using khmer-nltk ----
# Tokenize the raw text
# khmer-nltk's word_tokenize returns a list of tokens
tokens = word_tokenize(raw_text)

print(f'Total tokens: {len(tokens):,}')
print(f'First 20 tokens: {tokens[:20]}')


In [ ]:
# ---- 2. Build vocabulary (filter freq < 10, ignore spaces) ----

# Count token frequencies
token_counts = Counter(tokens)
print(f'Unique tokens before filtering: {len(token_counts):,}')

# Filter: keep tokens with frequency >= 10, exclude spaces
min_freq = 10
vocab = [
    token for token, count in token_counts.items()
    if count >= min_freq and token.strip() != '' and token != ' '
]
print(f'Vocabulary size after filtering (freq >= {min_freq}): {len(vocab):,}')

# Build mappings
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}
vocab_size = len(vocab)

# Filtered tokens (only keep words in vocab)
filtered_tokens = [t for t in tokens if t in word2idx]
print(f'Filtered tokens: {len(filtered_tokens):,}')
print(f'Vocabulary size: {vocab_size:,}')

In [ ]:
# ---- 3. Generate training pairs (center, context) ----

WINDOW_SIZE = 4  # ±4

def generate_skipgram_pairs(tokens, window_size):
    """Generate (center_idx, context_idx) pairs."""
    pairs = []
    indices = [word2idx[t] for t in tokens if t in word2idx]
    
    for i, center in enumerate(indices):
        # Context window: [i - window_size, i + window_size] excluding i
        start = max(0, i - window_size)
        end = min(len(indices), i + window_size + 1)
        for j in range(start, end):
            if j != i:
                context = indices[j]
                pairs.append((center, context))
    
    return pairs

pairs = generate_skipgram_pairs(filtered_tokens, WINDOW_SIZE)
print(f'Generated {len(pairs):,} training pairs')
print(f'Sample pairs (center -> context):')
for i in range(5):
    c, ctx = pairs[i]
    print(f'  {idx2word[c]} -> {idx2word[ctx]}')

In [ ]:
# ---- 4. Negative Sampling Implementation ----

# Compute unigram distribution (raised to 3/4 power)
NEG_SAMPLES = 2

# Get frequencies for words in vocabulary
freqs = np.array([token_counts[word] ** 0.75 for word in vocab], dtype=np.float64)
neg_sample_probs = freqs / freqs.sum()

def negative_sampling(batch_size, num_neg=NEG_SAMPLES):
    """Generate negative samples from the noise distribution."""
    neg_samples = np.random.choice(
        vocab_size, size=(batch_size, num_neg),
        p=neg_sample_probs
    )
    return torch.tensor(neg_samples, dtype=torch.long, device=device)

# ---- 5. Skip-gram Dataset ----

class SkipGramDataset(Dataset):
    def __init__(self, pairs):
        self.centers = torch.tensor([p[0] for p in pairs], dtype=torch.long)
        self.contexts = torch.tensor([p[1] for p in pairs], dtype=torch.long)
    
    def __len__(self):
        return len(self.centers)
    
    def __getitem__(self, idx):
        return self.centers[idx], self.contexts[idx]

dataset = SkipGramDataset(pairs)
dataloader = DataLoader(dataset, batch_size=512, shuffle=True)
print(f'Dataset size: {len(dataset):,} pairs')
print(f'Number of batches: {len(dataloader):,}')

In [ ]:
# ---- 6. Skip-gram Model ----

EMBEDDING_DIM = 50

class SkipGramModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.center_embeddings = nn.Embedding(vocab_size, embedding_dim)
        self.context_embeddings = nn.Embedding(vocab_size, embedding_dim)
        
        # Initialize embeddings
        init_range = 0.5 / embedding_dim
        nn.init.uniform_(self.center_embeddings.weight, -init_range, init_range)
        nn.init.uniform_(self.context_embeddings.weight, -init_range, init_range)
    
    def forward(self, center, context, negatives):
        """
        center: (batch_size,)
        context: (batch_size,)
        negatives: (batch_size, num_neg)
        """
        # Positive pair scores
        center_emb = self.center_embeddings(center)  # (batch, dim)
        context_emb = self.context_embeddings(context)  # (batch, dim)
        pos_scores = torch.sum(center_emb * context_emb, dim=1)  # (batch,)
        pos_loss = F.logsigmoid(pos_scores)
        
        # Negative pair scores
        neg_emb = self.context_embeddings(negatives)  # (batch, num_neg, dim)
        neg_scores = torch.bmm(
            neg_emb, center_emb.unsqueeze(2)  # (batch, num_neg, 1)
        ).squeeze(-1)  # (batch, num_neg)
        neg_loss = F.logsigmoid(-neg_scores).sum(dim=1)  # (batch,)
        
        return -(pos_loss + neg_loss).mean()

skipgram_model = SkipGramModel(vocab_size, EMBEDDING_DIM).to(device)
optimizer = optim.Adam(skipgram_model.parameters(), lr=0.001)

total_params = sum(p.numel() for p in skipgram_model.parameters())
print(f'Skip-gram model parameters: {total_params:,}')
print(f'Embedding matrix: {vocab_size} x {EMBEDDING_DIM}')

In [ ]:
# ---- 7. Train Skip-gram Model ----

NUM_EPOCHS = 5
LOG_INTERVAL = 200

skipgram_model.train()
losses = []

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_idx, (center, context) in enumerate(dataloader):
        center = center.to(device)
        context = context.to(device)
        batch_size = center.size(0)
        
        # Generate negative samples
        negatives = negative_sampling(batch_size)
        
        optimizer.zero_grad()
        loss = skipgram_model(center, context, negatives)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
        
        if (batch_idx + 1) % LOG_INTERVAL == 0:
            print(f'Epoch {epoch+1}/{NUM_EPOCHS}, Batch {batch_idx+1}/{len(dataloader)}, Loss: {loss.item():.4f}')
    
    avg_loss = epoch_loss / num_batches
    losses.append(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_EPOCHS} completed. Avg Loss: {avg_loss:.4f}')

print(f'\nTraining completed!')

In [ ]:
# ---- 8. Extract Skip-gram Embeddings ----

# Use the center embeddings as the final word vectors
skipgram_embeddings = skipgram_model.center_embeddings.weight.detach().cpu().numpy()
print(f'Saved embeddings shape: {skipgram_embeddings.shape}')

# Show some similar words
def find_similar_words(word, embeddings, idx2word, word2idx, top_k=10):
    if word not in word2idx:
        print(f'Word "{word}" not in vocabulary')
        return []
    idx = word2idx[word]
    vec = embeddings[idx]
    norms = np.linalg.norm(embeddings, axis=1)
    sims = (embeddings @ vec) / (norms * np.linalg.norm(vec) + 1e-8)
    top_indices = np.argsort(sims)[::-1][1:top_k+1]
    return [(idx2word[i], sims[i]) for i in top_indices]

# Test with some Khmer words
test_words = ['\u1794\u17d2\u179a\u17b6\u179f\u17b6\u1791', '\u17a2\u1784\u1782\u17d2\u179a', '\u179c\u178f\u17d2\u178f']
# These are: ប្រាសាទ, អង្គរ, វត្ត

for w in test_words:
    if w in word2idx:
        similar = find_similar_words(w, skipgram_embeddings, idx2word, word2idx)
        print(f'\nWords most similar to "{w}":')
        for sim_word, score in similar[:8]:
            print(f'  {sim_word}: {score:.4f}')

---
## Part II: PCA Visualization of Word Embeddings

In [ ]:
# ---- PCA Dimensionality Reduction ----

# Apply PCA to reduce 50D -> 2D
pca = PCA(n_components=2, random_state=SEED)
embeddings_2d = pca.fit_transform(skipgram_embeddings)

print(f'Explained variance ratio: {pca.explained_variance_ratio_}')
print(f'Total variance explained by 2 components: {pca.explained_variance_ratio_.sum():.4f}')

In [ ]:
# ---- 2D Plot of Word Embeddings ----

plt.figure(figsize=(14, 12))

# Only label a subset of words to avoid clutter
# Pick words with highest frequency
top_words = [w for w, _ in Counter(filtered_tokens).most_common(80)]
top_indices = [word2idx[w] for w in top_words if w in word2idx]

# Plot all words as small dots
plt.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    alpha=0.3, s=5, c='steelblue'
)

# Label the top words
for idx in top_indices:
    plt.annotate(
        idx2word[idx],
        (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
        fontsize=9, alpha=0.8,
        arrowprops=dict(arrowstyle='-', alpha=0.3)
    )

plt.title('2D PCA Projection of Skip-gram Word Embeddings (Part I)', fontsize=14)
plt.xlabel('Principal Component 1')
plt.ylabel('Principal Component 2')
plt.tight_layout()
plt.savefig('pca_skipgram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pca_skipgram.png')

---
## Part III: Neural Language Model

Predict the next word given **n=5 previous words**.
- Use pre-trained embeddings from Part I (fixed, not trained)
- Concatenate word embeddings: input size = 50 × 5 = 250
- Single hidden layer: size **512** with **sigmoid** activation
- Softmax output layer

In [ ]:
# ---- Prepare LM Dataset ----

N_PREV = 5  # Number of previous words to use

def generate_lm_data(tokens, n_prev):
    """Generate (context_indices, target_idx) for language model."""
    X, y = [], []
    indices = [word2idx[t] for t in tokens if t in word2idx]
    
    for i in range(n_prev, len(indices)):
        context = indices[i - n_prev:i]
        target = indices[i]
        X.append(context)
        y.append(target)
    
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

X_lm, y_lm = generate_lm_data(tokens, N_PREV)
print(f'LM dataset: {len(X_lm):,} samples')
print(f'Input shape: {X_lm.shape}')
print(f'Output shape: {y_lm.shape}')

In [ ]:
# ---- Neural Language Model (Fixed Embeddings) ----

HIDDEN_SIZE = 512

class NeuralLanguageModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, n_prev, hidden_size, pretrained_embeddings=None):
        super().__init__()
        self.n_prev = n_prev
        self.embedding_dim = embedding_dim
        
        # Embedding layer (initialized with pretrained or random)
        self.embeddings = nn.Embedding(vocab_size, embedding_dim)
        if pretrained_embeddings is not None:
            self.embeddings.weight.data.copy_(torch.tensor(pretrained_embeddings, dtype=torch.float32))
        
        # Input size = n_prev * embedding_dim (concatenated)
        input_size = n_prev * embedding_dim
        
        # Hidden layer with sigmoid
        self.hidden = nn.Linear(input_size, hidden_size)
        
        # Output layer
        self.output = nn.Linear(hidden_size, vocab_size)
    
    def forward(self, x):
        """
        x: (batch_size, n_prev) - indices of previous words
        """
        # Get embeddings and flatten
        emb = self.embeddings(x)  # (batch, n_prev, dim)
        emb = emb.view(emb.size(0), -1)  # (batch, n_prev*dim)
        
        # Hidden layer with sigmoid
        h = torch.sigmoid(self.hidden(emb))  # (batch, hidden_size)
        
        # Output layer (logits)
        logits = self.output(h)  # (batch, vocab_size)
        
        return logits

# Create model with FIXED (non-trainable) pretrained embeddings
lm_model_fixed = NeuralLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    n_prev=N_PREV,
    hidden_size=HIDDEN_SIZE,
    pretrained_embeddings=skipgram_embeddings
).to(device)

# Freeze the embedding layer
for param in lm_model_fixed.embeddings.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in lm_model_fixed.parameters() if p.requires_grad)
print(f'LM model trainable parameters: {total_params:,}')
print(f'Hidden layer: {N_PREV * EMBEDDING_DIM} -> {HIDDEN_SIZE} -> {vocab_size}')

In [ ]:
# ---- Train Neural Language Model (Fixed Embeddings) ----

lm_dataset = torch.utils.data.TensorDataset(X_lm, y_lm)
lm_dataloader = DataLoader(lm_dataset, batch_size=256, shuffle=True)

lm_optimizer = optim.Adam(lm_model_fixed.parameters(), lr=0.001)
lm_criterion = nn.CrossEntropyLoss()

NUM_LM_EPOCHS = 10

lm_model_fixed.train()
lm_losses = []

for epoch in range(NUM_LM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_x, batch_y in lm_dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        lm_optimizer.zero_grad()
        logits = lm_model_fixed(batch_x)
        loss = lm_criterion(logits, batch_y)
        loss.backward()
        lm_optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    lm_losses.append(avg_loss)
    
    # Compute perplexity
    perplexity = math.exp(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_LM_EPOCHS}, Loss: {avg_loss:.4f}, Perplexity: {perplexity:.2f}')

print(f'\nPart III training completed! Final perplexity: {math.exp(lm_losses[-1]):.2f}')

In [ ]:
# ---- Evaluate LM: Generate sample predictions ----

lm_model_fixed.eval()
with torch.no_grad():
    # Take a few samples from the dataset
    for i in range(5):
        context = X_lm[i:i+1].to(device)
        target = y_lm[i].item()
        
        logits = lm_model_fixed(context)
        probs = F.softmax(logits, dim=1)
        pred = logits.argmax(dim=1).item()
        
        context_words = ' '.join([idx2word[idx.item()] for idx in context[0]])
        print(f'Context: ...{context_words}...')
        print(f'  Actual next word: {idx2word[target]}')
        print(f'  Predicted next word: {idx2word[pred]}')
        print(f'  Top 5 candidates:')
        top5 = torch.topk(logits[0], 5)
        for j in range(5):
            w = idx2word[top5.indices[j].item()]
            p = torch.softmax(logits[0], dim=0)[top5.indices[j]].item()
            print(f'    {w}: {p:.4f}')
        print()

---
## Part IV: Learning Embeddings from Scratch

Modify the neural language model to **learn embeddings from scratch** during training (no pretrained embeddings, embeddings are trainable).

Then apply PCA and compare with the skip-gram embeddings from Part I.

In [ ]:
# ---- Neural Language Model (Learned Embeddings from Scratch) ----

lm_model_scratch = NeuralLanguageModel(
    vocab_size=vocab_size,
    embedding_dim=EMBEDDING_DIM,
    n_prev=N_PREV,
    hidden_size=HIDDEN_SIZE,
    pretrained_embeddings=None  # Random initialization
).to(device)

# All parameters are trainable (including embeddings)
total_params = sum(p.numel() for p in lm_model_scratch.parameters())
print(f'Scratch LM model total parameters: {total_params:,}')
print(f'Embeddings are trainable: {lm_model_scratch.embeddings.weight.requires_grad}')

In [ ]:
# ---- Train LM from Scratch ----

scratch_optimizer = optim.Adam(lm_model_scratch.parameters(), lr=0.001)
scratch_criterion = nn.CrossEntropyLoss()

lm_model_scratch.train()
scratch_losses = []

for epoch in range(NUM_LM_EPOCHS):
    epoch_loss = 0.0
    num_batches = 0
    
    for batch_x, batch_y in lm_dataloader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        
        scratch_optimizer.zero_grad()
        logits = lm_model_scratch(batch_x)
        loss = scratch_criterion(logits, batch_y)
        loss.backward()
        scratch_optimizer.step()
        
        epoch_loss += loss.item()
        num_batches += 1
    
    avg_loss = epoch_loss / num_batches
    scratch_losses.append(avg_loss)
    
    perplexity = math.exp(avg_loss)
    print(f'Epoch {epoch+1}/{NUM_LM_EPOCHS}, Loss: {avg_loss:.4f}, Perplexity: {perplexity:.2f}')

print(f'\nPart IV training completed! Final perplexity: {math.exp(scratch_losses[-1]):.2f}')

In [ ]:
# ---- Extract Embeddings Learned from Scratch ----

scratch_embeddings = lm_model_scratch.embeddings.weight.detach().cpu().numpy()
print(f'Scratch embeddings shape: {scratch_embeddings.shape}')

In [ ]:
# ---- PCA on Scratch Embeddings ----

pca_scratch = PCA(n_components=2, random_state=SEED)
scratch_embeddings_2d = pca_scratch.fit_transform(scratch_embeddings)

print(f'Scratch embeddings explained variance: {pca_scratch.explained_variance_ratio_}')
print(f'Total variance: {pca_scratch.explained_variance_ratio_.sum():.4f}')

In [ ]:
# ---- Comparison: Side-by-side PCA Plots ----

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# Get top words for labeling
top_words = [w for w, _ in Counter(filtered_tokens).most_common(60)]
top_indices = [word2idx[w] for w in top_words if w in word2idx]

# Plot 1: Skip-gram embeddings (Part I)
ax1 = axes[0]
ax1.scatter(
    embeddings_2d[:, 0], embeddings_2d[:, 1],
    alpha=0.3, s=5, c='steelblue'
)
for idx in top_indices:
    ax1.annotate(idx2word[idx], (embeddings_2d[idx, 0], embeddings_2d[idx, 1]),
                fontsize=8, alpha=0.7)
ax1.set_title('Part I: Skip-gram Embeddings (PCA)', fontsize=13)
ax1.set_xlabel('PC1')
ax1.set_ylabel('PC2')

# Plot 2: LM scratch embeddings (Part IV)
ax2 = axes[1]
ax2.scatter(
    scratch_embeddings_2d[:, 0], scratch_embeddings_2d[:, 1],
    alpha=0.3, s=5, c='coral'
)
for idx in top_indices:
    ax2.annotate(idx2word[idx], (scratch_embeddings_2d[idx, 0], scratch_embeddings_2d[idx, 1]),
                fontsize=8, alpha=0.7)
ax2.set_title('Part IV: LM Learned Embeddings (PCA)', fontsize=13)
ax2.set_xlabel('PC1')
ax2.set_ylabel('PC2')

plt.tight_layout()
plt.savefig('pca_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: pca_comparison.png')

In [ ]:
# ---- Similarity Comparison ----

print('=' * 60)
print('COMPARISON: Most Similar Words')
print('=' * 60)

for w in test_words:
    if w not in word2idx:
        continue
    
    similar_skipgram = find_similar_words(w, skipgram_embeddings, idx2word, word2idx, top_k=5)
    similar_scratch = find_similar_words(w, scratch_embeddings, idx2word, word2idx, top_k=5)
    
    print(f'\n--- "{w}" ({" ".join(w)} in Unicode) ---')
    print(f'  Skip-gram neighbors:')
    for sim_w, score in similar_skipgram:
        print(f'    {sim_w}: {score:.4f}')
    print(f'  LM Scratch neighbors:')
    for sim_w, score in similar_scratch:
        print(f'    {sim_w}: {score:.4f}')

# Compare cosine similarity between the two embedding spaces
# Align using orthogonal Procrustes or just compare cosine similarities
print(f'\n{"=" * 60}')
print('Cosine similarity between corresponding word vectors')
print(f'{"=" * 60}')

# Normalize embeddings
sg_norm = skipgram_embeddings / (np.linalg.norm(skipgram_embeddings, axis=1, keepdims=True) + 1e-8)
sc_norm = scratch_embeddings / (np.linalg.norm(scratch_embeddings, axis=1, keepdims=True) + 1e-8)

# Compute cosine similarity between corresponding vectors
cos_sims = np.sum(sg_norm * sc_norm, axis=1)
print(f'Mean cosine similarity: {cos_sims.mean():.4f}')
print(f'Median cosine similarity: {np.median(cos_sims):.4f}')
print(f'Std: {cos_sims.std():.4f}')
print(f'Top-5 most similar word vectors across models:')
top_sim_indices = np.argsort(cos_sims)[::-1][:5]
for idx in top_sim_indices:
    print(f'  {idx2word[idx]}: {cos_sims[idx]:.4f}')
print(f'Bottom-5 least similar:')
bottom_sim_indices = np.argsort(cos_sims)[:5]
for idx in bottom_sim_indices:
    print(f'  {idx2word[idx]}: {cos_sims[idx]:.4f}')

---
## Hyperparameter Tuning

Systematically compare different configurations:
- **Embedding dimensions:** [30, 50, 80, 100]
- **Window sizes:** [2, 4, 6]
- **Negative samples:** [1, 2, 5]

The skip-gram model is retrained for each configuration and evaluated by:
- Final training loss
- PCA variance explained (top 2 components)
- Cosine similarity between configs

In [ ]:
# ---- Hyperparameter Grid Search ----

EMBEDDING_DIMS = [30, 50, 80, 100]
WINDOW_SIZES = [2, 4, 6]
NEG_SAMPLES_LIST = [1, 2, 5]

def train_skipgram_and_get_embeddings(vocab_size, embedding_dim, window_size, neg_samples,
                                        pairs_list, freqs, num_epochs=5):
    """Train a skip-gram model and return embeddings + final loss."""
    # Reuse the global SkipGramModel class (defined earlier)
    
    dataset = torch.utils.data.TensorDataset(
        torch.tensor([p[0] for p in pairs_list], dtype=torch.long),
        torch.tensor([p[1] for p in pairs_list], dtype=torch.long)
    )
    loader = DataLoader(dataset, batch_size=512, shuffle=True)
    
    model = SkipGramModel(vocab_size, embedding_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    final_loss = 0.0
    for epoch in range(num_epochs):
        epoch_loss = 0.0
        for c, ctx in loader:
            c, ctx = c.to(device), ctx.to(device)
            neg = torch.tensor(
                np.random.choice(vocab_size, size=(c.size(0), neg_samples), p=freqs),
                dtype=torch.long, device=device
            )
            optimizer.zero_grad()
            loss = model(c, ctx, neg)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        final_loss = epoch_loss / len(loader)
    
    embeddings = model.center_embeddings.weight.detach().cpu().numpy()
    return embeddings, final_loss

print('Starting hyperparameter grid search...')
print(f'Configurations to test: {len(EMBEDDING_DIMS)} dims x {len(WINDOW_SIZES)} windows x {len(NEG_SAMPLES_LIST)} neg = {len(EMBEDDING_DIMS) * len(WINDOW_SIZES) * len(NEG_SAMPLES_LIST)}')

# Frequency distribution for negative sampling (use the existing one)
neg_probs = neg_sample_probs

results = []

for dim in EMBEDDING_DIMS:
    for ws in WINDOW_SIZES:
        for ns in NEG_SAMPLES_LIST:
            # Generate pairs for this window size
            pairs = generate_skipgram_pairs(filtered_tokens, ws)
            
            if len(pairs) == 0:
                continue
            
            print(f'  dim={dim}, window={ws}, neg={ns}... ', end='')
            emb, loss = train_skipgram_and_get_embeddings(
                vocab_size, dim, ws, ns, pairs, neg_probs
            )
            
            # Compute PCA variance
            pca = PCA(n_components=2, random_state=SEED)
            pca.fit(emb)
            pca_var = pca.explained_variance_ratio_.sum()
            
            results.append({
                'dim': dim, 'window': ws, 'neg': ns,
                'pairs': len(pairs),
                'loss': round(loss, 4),
                'pca_var_2d': round(pca_var, 4)
            })
            print(f'loss={loss:.4f}, pca_var={pca_var:.4f}')

print('\nGrid search completed!')

# Save results for later cells
tuning_results = results

In [ ]:
# ---- Display Results Table ----

df = pd.DataFrame(tuning_results)
print('=' * 70)
print('HYPERPARAMETER TUNING RESULTS')
print('=' * 70)
print()
print(df.to_string(index=False))

In [ ]:
# ---- Visualization: Effect of Each Hyperparameter ----

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Effect of embedding dimension (fixed window=4, neg=2)
base = [r for r in tuning_results if r['window'] == 4 and r['neg'] == 2]
if base:
    ax = axes[0]
    dims = [r['dim'] for r in base]
    losses = [r['loss'] for r in base]
    vars_ = [r['pca_var_2d'] for r in base]
    ax.plot(dims, losses, 'bo-', label='Loss')
    ax_twin = ax.twinx()
    ax_twin.plot(dims, vars_, 'rs--', label='PCA Var (2D)', alpha=0.7)
    ax.set_xlabel('Embedding Dimension')
    ax.set_ylabel('Loss', color='b')
    ax_twin.set_ylabel('PCA Var (2D)', color='r')
    ax.set_title('Effect of Embedding Dimension\n(window=4, neg=2)')
    ax.legend(loc='upper left')
    ax_twin.legend(loc='upper right')

# 2. Effect of window size (fixed dim=50, neg=2)
base2 = [r for r in tuning_results if r['dim'] == 50 and r['neg'] == 2]
if base2:
    ax = axes[1]
    ws = [r['window'] for r in base2]
    losses = [r['loss'] for r in base2]
    pairs = [r['pairs'] for r in base2]
    ax.plot(ws, losses, 'go-', label='Loss')
    ax_twin = ax.twinx()
    ax_twin.bar(ws, pairs, alpha=0.2, color='gray', label='# Pairs')
    ax.set_xlabel('Window Size')
    ax.set_ylabel('Loss', color='g')
    ax_twin.set_ylabel('# Training Pairs', color='gray')
    ax.set_title('Effect of Window Size\n(dim=50, neg=2)')
    ax.legend(loc='upper left')

# 3. Effect of negative samples (fixed dim=50, window=4)
base3 = [r for r in tuning_results if r['dim'] == 50 and r['window'] == 4]
if base3:
    ax = axes[2]
    ns = [r['neg'] for r in base3]
    losses = [r['loss'] for r in base3]
    vars_ = [r['pca_var_2d'] for r in base3]
    ax.plot(ns, losses, 'mo-', label='Loss')
    ax_twin = ax.twinx()
    ax_twin.plot(ns, vars_, 'cs--', label='PCA Var (2D)', alpha=0.7)
    ax.set_xlabel('Negative Samples')
    ax.set_ylabel('Loss', color='m')
    ax_twin.set_ylabel('PCA Var (2D)', color='c')
    ax.set_title('Effect of Negative Samples\n(dim=50, window=4)')
    ax.legend(loc='upper left')
    ax_twin.legend(loc='upper right')

plt.tight_layout()
plt.savefig('hyperparameter_tuning.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: hyperparameter_tuning.png')

In [ ]:
# ---- Best Configuration Summary ----

best_loss = min(tuning_results, key=lambda r: r['loss'])
best_pca = max(tuning_results, key=lambda r: r['pca_var_2d'])

print('=' * 60)
print('BEST CONFIGURATIONS')
print('=' * 60)
print()
print(f'Lowest loss: dim={best_loss["dim"]}, window={best_loss["window"]}, '
      f'neg={best_loss["neg"]} -> loss={best_loss["loss"]}')
print(f'Best PCA var: dim={best_pca["dim"]}, window={best_pca["window"]}, '
      f'neg={best_pca["neg"]} -> PCA 2D var={best_pca["pca_var_2d"]}')
print()
print('Observations:')
print('- Larger embedding dims typically capture more variance but need more data')
print('- Larger windows generate more training pairs but may introduce noise')
print('- More negative samples = more contrastive signal but slower training')
print()
print('RECOMMENDED: dim=100, window=6, neg=5 (for best coverage)')
print('CONSERVATIVE: dim=50, window=4, neg=2 (project default - faster, good balance)')

---
## Summary of Results

In [ ]:
print('=' * 60)
print('SUMMARY OF RESULTS')
print('=' * 60)
print(f'''
CORPUS STATISTICS
  Raw text: {len(raw_text):,} characters
  Raw tokens: {len(tokens):,}
  After filtering (freq >= 10): {len(filtered_tokens):,} tokens
  Vocabulary size: {vocab_size:,}

PART I: SKIP-GRAM
  Embedding dimension: {EMBEDDING_DIM}
  Context window: {WINDOW_SIZE}
  Negative samples: {NEG_SAMPLES}
  Training pairs: {len(pairs):,}

PART II: PCA VISUALIZATION
  Skip-gram PCA variance ratio: {pca.explained_variance_ratio_}
  Plot saved to: pca_skipgram.png

PART III: NEURAL LANGUAGE MODEL (Fixed Embeddings)
  Architecture: {N_PREV}*{EMBEDDING_DIM} -> {HIDDEN_SIZE} -> {vocab_size}
  Final loss: {lm_losses[-1]:.4f}
  Final perplexity: {math.exp(lm_losses[-1]):.2f}

PART IV: LM LEARNED FROM SCRATCH
  Architecture: {N_PREV}*{EMBEDDING_DIM} -> {HIDDEN_SIZE} -> {vocab_size}
  Final loss: {scratch_losses[-1]:.4f}
  Final perplexity: {math.exp(scratch_losses[-1]):.2f}
  Scratch PCA variance ratio: {pca_scratch.explained_variance_ratio_}
  Plot saved to: pca_comparison.png

CROSS-MODEL COMPARISON
  Mean cosine similarity between embeddings: {cos_sims.mean():.4f}
''')
print('=' * 60)

---
## Save All Trained Models

All models saved to \ directory.


In [ ]:
# ---- Save All Models to .pkl Files ----
import pickle
import os

os.makedirs('saved_models', exist_ok=True)
MODELS_DIR = 'saved_models'

# ---- 1. Vocabulary & Config ----
vocab_data = {
    'word2idx': word2idx,
    'idx2word': idx2word,
    'vocab': vocab,
    'vocab_size': vocab_size,
    'token_counts': dict(token_counts),
    'filtered_tokens': filtered_tokens,
    'raw_text': raw_text,
    'tokens': tokens,
    'neg_sample_probs': neg_sample_probs,
}
with open(f'{MODELS_DIR}/vocab_data.pkl', 'wb') as f: pickle.dump(vocab_data, f)
print(f'Saved: vocab_data.pkl')

# ---- 2. Config Parameters ----
config = {
    'EMBEDDING_DIM': EMBEDDING_DIM, 'WINDOW_SIZE': WINDOW_SIZE,
    'NEG_SAMPLES': NEG_SAMPLES, 'N_PREV': N_PREV,
    'HIDDEN_SIZE': HIDDEN_SIZE, 'NUM_EPOCHS': NUM_EPOCHS,
    'NUM_LM_EPOCHS': NUM_LM_EPOCHS, 'min_freq': min_freq,
    'SEED': SEED, 'device': str(device),
}
with open(f'{MODELS_DIR}/config.pkl', 'wb') as f: pickle.dump(config, f)
print(f'Saved: config.pkl')

# ---- 3. Skip-gram Model Checkpoint ----
torch.save({
    'model_state_dict': skipgram_model.state_dict(),    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'model_class': 'SkipGramModel',}, f'{MODELS_DIR}/skipgram_model.pt')
print(f'Saved: skipgram_model.pt')

# ---- 4. Skip-gram Embeddings (numpy) ----
with open(f'{MODELS_DIR}/skipgram_embeddings.pkl', 'wb') as f:
    pickle.dump(skipgram_embeddings, f)
print(f'Saved: skipgram_embeddings.pkl')

# ---- 5. Neural LM (Fixed) Checkpoint ----
torch.save({
    'model_state_dict': lm_model_fixed.state_dict(),    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'n_prev': N_PREV,
    'hidden_size': HIDDEN_SIZE,
    'model_class': 'NeuralLanguageModel',    'has_pretrained': True,
    'final_perplexity': math.exp(lm_losses[-1]),}, f'{MODELS_DIR}/lm_model_fixed.pt')
print(f'Saved: lm_model_fixed.pt')

# ---- 6. Neural LM (Scratch) Checkpoint ----
torch.save({
    'model_state_dict': lm_model_scratch.state_dict(),    'vocab_size': vocab_size,
    'embedding_dim': EMBEDDING_DIM,
    'n_prev': N_PREV,
    'hidden_size': HIDDEN_SIZE,
    'model_class': 'NeuralLanguageModel',    'has_pretrained': False,
    'final_perplexity': math.exp(scratch_losses[-1]),}, f'{MODELS_DIR}/lm_model_scratch.pt')
print(f'Saved: lm_model_scratch.pt')

# ---- 7. Scratch Embeddings (numpy) ----
with open(f'{MODELS_DIR}/scratch_embeddings.pkl', 'wb') as f:
    pickle.dump(scratch_embeddings, f)
print(f'Saved: scratch_embeddings.pkl')

# ---- 8. PCA Objects ----
pca_data = {
    'pca_skipgram': pca,
    'pca_scratch': pca_scratch,
    'embeddings_2d': embeddings_2d,
    'scratch_embeddings_2d': scratch_embeddings_2d,
}
with open(f'{MODELS_DIR}/pca_data.pkl', 'wb') as f: pickle.dump(pca_data, f)
print(f'Saved: pca_data.pkl')

# ---- 9. Tuning Results ----
with open(f'{MODELS_DIR}/tuning_results.pkl', 'wb') as f:
    pickle.dump(tuning_results, f)
print(f'Saved: tuning_results.pkl')

# ---- 10. LM Data ----
lm_data = {'X_lm': X_lm, 'y_lm': y_lm}
with open(f'{MODELS_DIR}/lm_data.pkl', 'wb') as f: pickle.dump(lm_data, f)
print(f'Saved: lm_data.pkl')

# ---- 11. Comparison Data ----
comparison_data = {
    'cos_sims': cos_sims,
    'mean_cosine_sim': float(cos_sims.mean()),
}
with open(f'{MODELS_DIR}/comparison_data.pkl', 'wb') as f: pickle.dump(comparison_data, f)
print(f'Saved: comparison_data.pkl')

# ---- Summary ----
import glob
print('\n' + '=' * 50)
print('ALL MODELS SAVED SUCCESSFULLY')
print('=' * 50)
for f in sorted(glob.glob(f'{MODELS_DIR}/*')):
    sz = os.path.getsize(f)
    print(f'  {os.path.basename(f)}: {sz/1024:.1f} KB')
